---
**Script name:** 05_bat_data_daytime_filtering.ipynb

**Purpose of script:** This script is to filter data to keep only nighttime active records. 

**Author:** Dr Katie Allan

**Date Created:** 2023-02-21

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Ensure folder paths are set properly.

---

## Import libraries that are required

In [1]:
import time
from datetime import timedelta
import sqlalchemy as psql
import pickle
import pandas as pd
import numpy as np
import os

## Functions that are used throughout the script

In [14]:
def bat_sort(dataframe, bat_column):
    #create empty python diction ary
    bat_dic = {}
    
    #create list of unique bat tags 
    unique_bats = sorted(dataframe[bat_column].unique())
    
    for bat in range(len(unique_bats)):
        #set bat name
        bat_name = unique_bats[bat]
        
        #filter dataframe by bat name
        bat_dataframe = dataframe[dataframe[bat_column] == bat_name].sort_values(by = 'time')
        
        #assign to dictionary
        bat_dic[bat_name] = bat_dataframe
        
    return bat_dic

In [19]:
# Initialize a dictionary to store the filtered DataFrame
def filter_dates(dictionary, sunlight_data):
    sunlight_data = sunlight_data[['Date', 'sunriseEnd', 'sunsetStart']].drop_duplicates().copy()
    dates_to_filter = {}
    
    for id, df in dictionary.items():
        #create new column of daytime detections
        df['daytime_detection'] = (
            (df['daytime'] >= df['sunrise']) & 
            (df['daytime'] <= df['sunset'])
        ).astype(int)
    
        # Group the DataFrame by the 'Date' column
        grouped = df.groupby(['Date'])['daytime_detection'].any()
        days_with_daytime_detection = grouped[grouped].index.to_frame().reset_index(drop=True)
        
        # Merge with sunrise data for current date
        days_with_daytime_detection = days_with_daytime_detection.merge(
            sunlight_data[['Date', 'sunriseEnd']],
            left_on='Date',
            right_on='Date',
            how='left'
        ).rename(columns={'sunriseEnd': 'Date_sunrise'})

        # Merge with sunrise data for the next day
        days_with_daytime_detection['Next_Date'] = days_with_daytime_detection['Date'] + pd.Timedelta(days=1)
        days_with_daytime_detection = days_with_daytime_detection.merge(
            sunlight_data[['Date', 'sunriseEnd']],
            left_on='Next_Date',
            right_on='Date',
            how='left',
            suffixes=('', '_next')
        ).rename(columns={'sunriseEnd': 'Next_Date_sunrise'})
    
        #Store the result in the dictionary
        dates_to_filter[id] = days_with_daytime_detection

        print(f"Processed {id}")

    return dates_to_filter

## Define variables

In [17]:
# Create a dictionary that maps keys to filter conditions
filtering_conditions = {
    'Kay': lambda df: df[df['time'] <= "2023-10-03 06:00:00"],
    'Larry': lambda df: df[df['time'] <= "2022-10-11 00:00:00"],
    'Mary': lambda df:df[df['time'] <= "2022-10-26 12:00:00"],
    'May': lambda df: df[df['time'] <= "2023-10-02 17:00:00"],
    'Ross': lambda df: df[df['time'] <= "2023-10-09 00:00:00"],
    'Strachy': lambda df: df[df['time'] <= "2023-10-13 00:00:00"],
    'Usher': lambda df: df[df['time'] <= "2023-01-05 07:00:00"],
    'Xanthi': lambda df: df[df['time'] <= "2023-03-14 22:00:00"]
}

## Load Data

In [ ]:
#load csvs of data
bat_data = pd.read_csv('output/bat_data_filtered_classification_joined_reduced.csv')
sun_data = pd.read_csv('data/sun_data.csv')
bat_tag_data = pd.read_csv('output/bat_tags.csv')

In [4]:
#show the top rows of the dataframe
bat_data

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,method,data_size,median_error_seconds,...,ring_no,capture_site,Date,daytime,week,max_rssi,n_receivers,activity_class,rec_type,window_period
0,5,19334B78,3750b3,-85.0,0,2023-02-16 12:49:19+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:19,7,NaN,NaN,none,Node,None
1,5,19334B78,3750b3,-88.0,0,2023-02-16 12:49:21+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:21,7,NaN,NaN,none,Node,None
2,5,19334B78,3750b3,-93.0,0,2023-02-16 12:49:23+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:23,7,NaN,NaN,none,Node,None
3,5,19334B78,3750b3,-73.0,0,2023-02-16 12:49:25+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:25,7,NaN,NaN,none,Node,None
4,5,19334B78,3750b3,-80.0,0,2023-02-16 12:49:36+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:36,7,NaN,NaN,none,Node,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191249,5,4C2A4B4B,3750b3,-67.0,0,2023-02-20 23:55:37+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:55:37,8,NaN,NaN,none,Node,None
1191250,5,4C2A4B4B,3750b3,-66.0,0,2023-02-20 23:55:47+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:55:47,8,NaN,NaN,none,Node,None
1191251,5,4C2A4B4B,3750b3,-67.0,0,2023-02-20 23:55:58+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:55:58,8,NaN,NaN,none,Node,None
1191252,5,4C2A4B4B,3750b3,-66.0,0,2023-02-20 23:56:08+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:56:08,8,NaN,NaN,none,Node,None


In [5]:
sun_data

,Date,lat,lon,sunrise,sunriseEnd,sunsetStart,sunset,dusk,night,nadir,nightEnd,dawn
0,2021-09-13,50.590305,-3.61261,2021-09-13 05:48:17+00:00,2021-09-13 05:51:40+00:00,2021-09-13 18:31:57+00:00,2021-09-13 18:35:20+00:00,2021-09-13 19:08:25+00:00,2021-09-13 20:30:03+00:00,2021-09-13 00:11:48+00:00,2021-09-13 03:53:33+00:00,2021-09-13 05:15:12+00:00
1,2021-09-14,50.590305,-3.61261,2021-09-14 05:49:48+00:00,2021-09-14 05:53:11+00:00,2021-09-14 18:29:42+00:00,2021-09-14 18:33:05+00:00,2021-09-14 19:06:06+00:00,2021-09-14 20:27:22+00:00,2021-09-14 00:11:26+00:00,2021-09-14 03:55:31+00:00,2021-09-14 05:16:46+00:00
2,2021-09-15,50.590305,-3.61261,2021-09-15 05:51:19+00:00,2021-09-15 05:54:41+00:00,2021-09-15 18:27:27+00:00,2021-09-15 18:30:50+00:00,2021-09-15 19:03:48+00:00,2021-09-15 20:24:42+00:00,2021-09-15 00:11:04+00:00,2021-09-15 03:57:27+00:00,2021-09-15 05:18:21+00:00
3,2021-09-16,50.590305,-3.61261,2021-09-16 05:52:50+00:00,2021-09-16 05:56:12+00:00,2021-09-16 18:25:12+00:00,2021-09-16 18:28:34+00:00,2021-09-16 19:01:29+00:00,2021-09-16 20:22:02+00:00,2021-09-16 00:10:42+00:00,2021-09-16 03:59:22+00:00,2021-09-16 05:19:55+00:00
4,2021-09-17,50.590305,-3.61261,2021-09-17 05:54:21+00:00,2021-09-17 05:57:44+00:00,2021-09-17 18:22:57+00:00,2021-09-17 18:26:19+00:00,2021-09-17 18:59:11+00:00,2021-09-17 20:19:24+00:00,2021-09-17 00:10:20+00:00,2021-09-17 04:01:16+00:00,2021-09-17 05:21:30+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
860,2024-01-21,50.590305,-3.61261,2024-01-21 08:05:52+00:00,2024-01-21 08:09:49+00:00,2024-01-21 16:43:31+00:00,2024-01-21 16:47:28+00:00,2024-01-21 17:24:21+00:00,2024-01-21 18:43:42+00:00,2024-01-21 00:26:40+00:00,2024-01-21 06:09:38+00:00,2024-01-21 07:28:59+00:00
861,2024-01-22,50.590305,-3.61261,2024-01-22 08:04:50+00:00,2024-01-22 08:08:46+00:00,2024-01-22 16:45:09+00:00,2024-01-22 16:49:06+00:00,2024-01-22 17:25:51+00:00,2024-01-22 18:45:03+00:00,2024-01-22 00:26:58+00:00,2024-01-22 06:08:52+00:00,2024-01-22 07:28:04+00:00
862,2024-01-23,50.590305,-3.61261,2024-01-23 08:03:44+00:00,2024-01-23 08:07:39+00:00,2024-01-23 16:46:49+00:00,2024-01-23 16:50:44+00:00,2024-01-23 17:27:22+00:00,2024-01-23 18:46:25+00:00,2024-01-23 00:27:14+00:00,2024-01-23 06:08:03+00:00,2024-01-23 07:27:07+00:00
863,2024-01-24,50.590305,-3.61261,2024-01-24 08:02:37+00:00,2024-01-24 08:06:31+00:00,2024-01-24 16:48:30+00:00,2024-01-24 16:52:24+00:00,2024-01-24 17:28:54+00:00,2024-01-24 18:47:48+00:00,2024-01-24 00:27:30+00:00,2024-01-24 06:07:12+00:00,2024-01-24 07:26:06+00:00


In [6]:
#check column types of each variable
bat_data.dtypes

radio_id                             object
tag_id                               object
node_id                              object
tag_rssi                            float64
validated                             int64
time                    datetime64[ns, UTC]
station_id                           object
method                               object
data_size                           float64
median_error_seconds                float64
max_error_seconds                   float64
latitude                            float64
longitude                           float64
deploy                  datetime64[ns, UTC]
remove                  datetime64[ns, UTC]
location                             object
software                             object
bat_name                             object
ring_no                              object
capture_site                         object
Date                         datetime64[ns]
daytime                              object
week                            

In [7]:
sun_data.dtypes

Date                datetime64[ns]
lat                        float64
lon                        float64
sunrise        datetime64[ns, UTC]
sunriseEnd     datetime64[ns, UTC]
sunsetStart    datetime64[ns, UTC]
sunset         datetime64[ns, UTC]
dusk           datetime64[ns, UTC]
night          datetime64[ns, UTC]
nadir          datetime64[ns, UTC]
nightEnd       datetime64[ns, UTC]
dawn           datetime64[ns, UTC]
dtype: object

## Data processing

In [3]:
# remove any duplicate rows from dataframe
bat_data = bat_data.drop_duplicates()
sun_data = sun_data.drop_duplicates()
bat_tag_data = bat_tag_data.drop_duplicates()

In [8]:
#Change timezone of deploy and remove
bat_data['deploy'] = bat_data['deploy'].dt.tz_convert('Europe/London')
bat_data['remove'] = bat_data['deploy'].dt.tz_convert('Europe/London')

In [9]:
#create new date and daytime columns
bat_data['Date'] =  pd.to_datetime(bat_data['time'].dt.date, format='%Y-%m-%d', errors='coerce')
bat_data['daytime'] = bat_data['time'].dt.time

In [10]:
#Merge sunrise and sunset columns to bat data by Date
bat_data = pd.merge(left = bat_data,
                    right = sun_data[['Date', 'sunriseEnd', 'sunsetStart']],
                    how = 'left',
                   left_on = 'Date',
                   right_on = 'Date')

In [11]:
#change sunrise and sunset time to jsut time
bat_data['sunrise'] = bat_data['sunriseEnd'].dt.time
bat_data['sunset'] = bat_data['sunsetStart'].dt.time
bat_data = bat_data.drop(columns=['sunriseEnd', 'sunsetStart'])

In [ ]:
# Remove any data that was marked to be removed by rolling window
bat_data = bat_data[bat_data['window_period'] != 'remove']

In [15]:
#sort bats into a dictionary
bat_dic = bat_sort(bat_data, 'bat_name')

In [18]:
# Apply the conditions using dictionary comprehension to remove rows that are wrong
bat_dic_filtered = {key: filtering_conditions.get(key, lambda df: df)(df) for key, df in bat_dic.items()}

In [20]:
# create dictionary of times to filter per bat
times_to_filter = filter_dates(bat_dic_filtered, sun_data)

Processed Abby
Processed Andrea
Processed Bert
Processed Blob
Processed Charlie
Processed Conchita
Processed Derek
Processed Erik
Processed Fred
Processed George
Processed Gerty
Processed Heather
Processed Ian
Processed Iris
Processed Isla
Processed Jennet
Processed Jez
Processed Joyce
Processed Kay
Processed Larry
Processed Laticia
Processed Mary
Processed May
Processed Moira
Processed Nancy
Processed Neil
Processed Norman
Processed Olga
Processed Olivia
Processed Pedro
Processed Polly
Processed Queenie
Processed Quincey
Processed Rex
Processed Ross
Processed Steve
Processed Strachy
Processed Tony
Processed Tracy
Processed Uri
Processed Usher
Processed Vera
Processed Vincent
Processed Wally
Processed Wilma
Processed Xanthi
Processed Xray
Processed Yasmin
Processed Yvette
Processed Zac
Processed Zeus


C:\Users\allan\AppData\Local\Temp\ipykernel_19904\2007870348.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['daytime_detection'] = (
C:\Users\allan\AppData\Local\Temp\ipykernel_19904\2007870348.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['daytime_detection'] = (
C:\Users\allan\AppData\Local\Temp\ipykernel_19904\2007870348.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

In [21]:
# remove empty dataframes from dictionary
times_to_filter = {key:df for (key,df) in times_to_filter.items() if not df.empty}

## Filter to remove any daytime detections

In [22]:
# create blank dictionary
bat_daytime_data = {}

# loop through times to be filtered 
for (key, f_df) in times_to_filter.items():
    if key in bat_dic_filtered.keys():
        # create blank list
        n_bat = []
        #print(key)
        
        #loop through each row of times to be filtered
        for ind in f_df.index:
            # filter bat data by sunrise times, keep days where daytime detections
            d_bat = bat_dic_filtered[key][
            ((bat_dic_filtered[key]['time'] >= f_df['Date_sunrise'][ind]) &
             (bat_dic_filtered[key]['time'] < f_df['Next_Date_sunrise'][ind]))
            ].copy()
            
            # add column based on 'bat day'
            d_bat['bat_day'] = f_df['Date'].iloc[ind]
            d_bat['next_bat_day'] = f_df['Date_next'].iloc[ind]
            
            # add to blank list
            n_bat.append(d_bat)

    # add to dictionary with key
    bat_daytime_data[key] = pd.concat(n_bat, ignore_index = True)

In [23]:
# create blank dictionary
bat_nighttime_data = {}

# loop through times to be filtered 
for (key, f_df) in bat_dic_filtered.items():
    #print(key)
    if key in bat_daytime_data.keys():
        # outer join dataframes
        bat_all = f_df.merge(bat_daytime_data[key].drop_duplicates(),
                   how = 'left', indicator=True)
        
        # filter to keep only nighttime detections
        bat_night_only = bat_all[bat_all['_merge'] == 'left_only']
        
        #drop '_merge' column
        bat_night_only = bat_night_only.drop('_merge', axis=1)
        
        # add to dictionary with key
        bat_nighttime_data[key] = bat_night_only
        
    else:
        # add to dictionary with key - no changes
        bat_nighttime_data[key] = f_df

In [24]:
# check that the previoius function worked
for (key, df) in bat_nighttime_data.items():
    if (df['daytime_detection'] == 1).any():
        print(f'Dataframe for {key} includes daytime detections')
    else:
        print(f'Dataframe for {key} includes only nighttime detections')

Dataframe for Abby includes only nighttime detections
Dataframe for Andrea includes only nighttime detections
Dataframe for Bert includes only nighttime detections
Dataframe for Blob includes only nighttime detections
Dataframe for Charlie includes only nighttime detections
Dataframe for Conchita includes only nighttime detections
Dataframe for Derek includes only nighttime detections
Dataframe for Erik includes only nighttime detections
Dataframe for Fred includes only nighttime detections
Dataframe for George includes only nighttime detections
Dataframe for Gerty includes only nighttime detections
Dataframe for Heather includes only nighttime detections
Dataframe for Ian includes only nighttime detections
Dataframe for Iris includes only nighttime detections
Dataframe for Isla includes only nighttime detections
Dataframe for Jennet includes only nighttime detections
Dataframe for Jez includes only nighttime detections
Dataframe for Joyce includes only nighttime detections
Dataframe f

## Check that bat actually left the roost in nightime detections (compare to receiver ID's from daytime)

In [25]:
# loop through daytime detection dictionary
# Create dictionary to store the final filtered dataframes and pivot table of bat daytime detection counts
filtered_bat_nighttime_data = {}
daytime_locations_count = {}

# Loop through dictionary of bat daytime detections
for (key, df) in bat_daytime_data.items():
    # print the bat name
    print(key)
    
    # create blank list to store the filtered nighttime results
    filtered_bat_all_nights = []
    
    # sort_data by time
    sorted_bat_data = df.sort_values(by='time')
    
    # filter to keep only daytime detections of all data from one bat
    daytime_data = sorted_bat_data[sorted_bat_data['daytime_detection'] == 1].reset_index(drop=True)
    
    # create table count of daytime detections of all data from one bat
    daytime_data_location_count = pd.pivot_table(daytime_data, index= ['Date'], columns = ['location'], values = ['bat_name'], aggfunc = 'count', fill_value = 0)
    
    # add to dictionary of daytime location counts
    daytime_locations_count[key] = daytime_data_location_count

    # set list of unique bat days for individual bat
    unique_bat_days = sorted_bat_data['bat_day'].unique()
    
    # loop through each 'bat_day'
    for day in unique_bat_days:
        # filter sorted data by 'bat_day' and only daytime detections 
        daytime_data = sorted_bat_data[(sorted_bat_data['bat_day'] == day) & (sorted_bat_data['daytime_detection'] == 1)].reset_index(drop=True)
        nighttime_data = sorted_bat_data[(sorted_bat_data['bat_day'] == day) & (sorted_bat_data['daytime_detection'] == 0)].reset_index(drop=True)

        # If nighttime_data is empty, skip to the next day
        if nighttime_data.empty:
            print(f"No nighttime data for {key} on day {day}, skipping to next day.")
            continue
        
        # set the day of year, next day of year and list of unique daytime locations
        doy = nighttime_data['bat_day'].iloc[0]
        next_doy = nighttime_data['next_bat_day'].iloc[0]
        unique_day_locations = daytime_data['location'].unique()
        
        # print the day and date
        print(f'Checking data on {doy} for different location detection.') 
        
        # Initialize variables
        filtered_bat_individual_night = None
        bat_emergence_time = None
        bat_home_time = None
        
        # Find the first nighttime detection with a different location to daytime detections
        for idx, row in nighttime_data.iterrows():
            # check if the detection of that row matches the locations of the daytime detections
            if row['location'] not in unique_day_locations:
                    # If the location of the row is different, select as bat emergence time
                    bat_emergence_time = idx
                    break
            else:
                # The bat didn't leave the roost as all nighttime records match the daytime records
                bat_emergence_time = 0

        # If the bat emerged, filter the nighttime data accordingly
        if bat_emergence_time != 0:
            # Set the default home time to the end of the night
            bat_home_time = len(nighttime_data)
            # If the next day in year has daytime detections, find the first record of the bat at these receivers
            if next_doy in unique_bat_days:
                # Set list of unique daytime locations for next day of year
                unique_next_day_locations = sorted_bat_data[(sorted_bat_data['bat_day'] == next_doy) & (sorted_bat_data['daytime_detection'] == 1)]['location'].unique()
                # Reverse iterate through nighttime data
                for idx, row in nighttime_data.iloc[::-1].iterrows():
                    # find the first record of the next day of year daytime location data
                    if row['location'] not in unique_next_day_locations:
                        # set the final index
                        bat_home_time = idx + 1
                        break
            # Filter data from bat emergence time to bat home time
            filtered_bat_individual_night = nighttime_data.iloc[bat_emergence_time:bat_home_time]

        # If the bat emerged from the roost, add to the list of dataframes
        if filtered_bat_individual_night is not None:
            filtered_bat_all_nights.append(filtered_bat_individual_night)
        # If the bat stayed at home, print message
        else:
            print(f"No nonmatching nighttime location found for {key} on {doy}. The bat  did not leave the roost.")  
        
    # Add dataframe to dictionary with key of bat name
    if filtered_bat_all_nights:
        filtered_bat_nighttime_data[key] = pd.concat(filtered_bat_all_nights, ignore_index=True)
    else:
        print(f"No nighttime data added for {key}.")

Abby
No nighttime data for Abby on day 2023-02-16 00:00:00, skipping to next day.
Checking data on 2023-02-17 00:00:00 for different location detection.
No nonmatching nighttime location found for Abby on 2023-02-17 00:00:00. The bat  did not leave the roost.
No nighttime data added for Abby.
Andrea
Checking data on 2023-11-14 00:00:00 for different location detection.
Checking data on 2023-11-15 00:00:00 for different location detection.
Checking data on 2023-11-17 00:00:00 for different location detection.
No nonmatching nighttime location found for Andrea on 2023-11-17 00:00:00. The bat  did not leave the roost.
Checking data on 2023-11-18 00:00:00 for different location detection.
Checking data on 2023-11-19 00:00:00 for different location detection.
Checking data on 2023-11-20 00:00:00 for different location detection.
Checking data on 2023-11-21 00:00:00 for different location detection.
Checking data on 2023-11-22 00:00:00 for different location detection.
Checking data on 2023-

In [26]:
# create blank dictionary
all_bat_night_dictionary = {}

# loop through times to be filtered 
for (key, f_df) in bat_nighttime_data.items():
    if key in filtered_bat_nighttime_data.keys():
        #join datframes by matching the keys
        concat_dfs = pd.concat([f_df, filtered_bat_nighttime_data[key]], axis=0, ignore_index=True)
        
        #add to dictionary with key
        all_bat_night_dictionary[key] = concat_dfs
    else:
        #add to dicionary with key
        all_bat_night_dictionary[key] = f_df

all_bat_night_dictionary

{'Abby': Empty DataFrame
 Columns: [radio_id, tag_id, node_id, tag_rssi, validated, time, station_id, method, data_size, median_error_seconds, max_error_seconds, latitude, longitude, deploy, remove, location, software, bat_name, ring_no, capture_site, Date, daytime, week, max_rssi, n_receivers, activity_class, rec_type, window_period, sunrise, sunset, daytime_detection, bat_day, next_bat_day]
 Index: []
 
 [0 rows x 33 columns],
 'Andrea':      radio_id    tag_id node_id  tag_rssi  validated  \
 0           5  33663378  3781a5    -107.0          0   
 1           5  33663378  3781a5    -109.0          0   
 2           5  33663378  3781a5    -106.0          0   
 3           5  33663378  3781a5    -109.0          0   
 4           5  33663378  328787    -108.0          0   
 ...       ...       ...     ...       ...        ...   
 3934        5  33663378  3292c8     -95.0          0   
 3935        5  33663378  3292c8    -102.0          0   
 3936        5  33663378  328787    -108.0  

In [27]:
#turn back into one big dataframe
all_bat_night_dataframe = pd.concat(all_bat_night_dictionary.values(), ignore_index=True)
all_bat_night_dataframe

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,method,data_size,median_error_seconds,...,max_rssi,n_receivers,activity_class,rec_type,window_period,sunrise,sunset,daytime_detection,bat_day,next_bat_day
0,5,33663378,3781a5,-107.0,0,2023-11-10 18:54:13+00:00,None,kl_div,29837.0,123.0,...,NaN,NaN,none,Node,None,07:25:07,16:34:13,0,NaT,NaT
1,5,33663378,3781a5,-109.0,0,2023-11-10 18:54:23+00:00,None,kl_div,29837.0,123.0,...,NaN,NaN,none,Node,None,07:25:07,16:34:13,0,NaT,NaT
2,5,33663378,3781a5,-106.0,0,2023-11-10 18:54:33+00:00,None,kl_div,29837.0,123.0,...,NaN,NaN,none,Node,None,07:25:07,16:34:13,0,NaT,NaT
3,5,33663378,3781a5,-109.0,0,2023-11-10 18:54:53+00:00,None,kl_div,29837.0,123.0,...,NaN,NaN,none,Node,None,07:25:07,16:34:13,0,NaT,NaT
4,5,33663378,328787,-108.0,0,2023-11-10 18:56:44+00:00,None,kl_div,7509.0,123.0,...,NaN,NaN,none,Node,None,07:25:07,16:34:13,0,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144373,1,2A615252,None,-108.0,0,2023-12-31 00:42:52+00:00,31D096AEC921,ctt,NaN,NaN,...,-88.0,1.0,active,Station,None,08:21:13,16:15:31,0,NaT,NaT
144374,1,2A615252,None,-103.0,0,2023-12-31 00:43:02+00:00,31D096AEC921,ctt,NaN,NaN,...,-103.0,1.0,active,Station,None,08:21:13,16:15:31,0,NaT,NaT
144375,1,2A615252,None,-96.0,0,2023-12-31 00:45:09+00:00,31D096AEC921,ctt,NaN,NaN,...,-96.0,1.0,active,Station,None,08:21:13,16:15:31,0,NaT,NaT
144376,1,2A615252,None,-99.0,0,2023-12-31 00:45:19+00:00,31D096AEC921,ctt,NaN,NaN,...,-96.0,1.0,active,Station,None,08:21:13,16:15:31,0,NaT,NaT


In [28]:
len(all_bat_night_dataframe)

144378

In [29]:
len(bat_data) - len(all_bat_night_dataframe)

309160

## Export data

In [ ]:
#export data to csv file
all_bat_night_dataframe.to_csv('output/bat_data_filtered_classification_joined_reduced_nighttime.csv', index=False)